# Interactive Multi-Condition Overlap Heatmap

This notebook demonstrates how to visualize the overlap of multiple conditions in a 2D parameter space using **Plotly**.

It uses a generalized function `plot_condition_heatmap` to dynamically pass arguments to condition functions, and automatically formats hover labels based on threshold kwargs.

In [18]:
import sys
import os
import inspect
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from scipy.signal import find_peaks

# Add project path to import helpers
project_dir = "/home/pseudonym/code/Nonlocal_Conductance/Nonlocality-of-local-Andreev-conductances-as-a-probe-for-topological-Majorana-wires"
if project_dir not in sys.path:
    sys.path.append(project_dir)
import helpers as hp

In [19]:
# Setup data directory
dirname = Path(project_dir) / "Data/one_disorder/disorder_realization_00"

# Load conductances and parameters
brcl = np.load(dirname / "barrier_right_conductance_left_arr.npy")
brcr = np.load(dirname / "barrier_right_conductance_right_arr.npy")
peaks_left = np.load(dirname / "peaks_left.npy")
peaks_right = np.load(dirname / "peaks_right.npy")
params_list = np.load(dirname / "params_list.npy")

mu = params_list[:, 1]
V_z = params_list[:, 2]


In [20]:
# Master plotting function
def plot_condition_heatmap(conditions_dict, data_dict):
    """
    Dynamically checks conditions and plots a plotly heatmap of their overlaps.
    
    Args:
        conditions_dict: dict of { 'Label_template': condition_function }
        data_dict: dict of { 'arg_name': data_array_or_kwarg }
    """
    mu_arr = data_dict['mu']
    vz_arr = data_dict['V_z']
    
    num_points = len(mu_arr)
    condition_results = {} # Store binary arrays for each condition
    formatted_labels = []
    
    # Evaluate each condition dynamically
    for label_template, func in conditions_dict.items():
        sig = inspect.signature(func)
        kwargs = {}
        for param_name in sig.parameters:
            if param_name in data_dict:
                kwargs[param_name] = data_dict[param_name]
            else:
                raise ValueError(f"Missing required argument '{param_name}' for function '{func.__name__}' in data_dict.")
        
        # Format label with kwargs (like threshold parameters)
        label = label_template.format(**kwargs)
        formatted_labels.append(label)
        
        res = func(**kwargs)
        condition_results[label] = np.asarray(res).astype(bool)
    
    # Create a sum of conditions met
    z_sum = np.zeros(num_points, dtype=int)
    for label, res in condition_results.items():
        z_sum += res.astype(int)
        
    # Build hover texts
    hover_text = []
    for i in range(num_points):
        met = [label for label in formatted_labels if condition_results[label][i]]
        met_str = "<br>  - " + "<br>  - ".join(met) if met else "None"
        hover_text.append(
            f"<b>V_z:</b> {vz_arr[i]:.3f} meV<br>"
            f"<b>mu:</b> {mu_arr[i]:.3f} meV<br>"
            f"<b>Conditions Met:</b> {z_sum[i]}/{len(formatted_labels)}<br>"
            f"<b>Details:</b> {met_str}"
        )
        
    # Reshape for Heatmap
    mu_vals = np.unique(mu_arr)
    vz_vals = np.unique(vz_arr)
    n_mu = len(mu_vals)
    n_vz = len(vz_vals)
    
    Z_2d = z_sum.reshape(n_mu, n_vz)
    hover_text_2d = np.array(hover_text).reshape(n_mu, n_vz)
    
    max_conditions = len(formatted_labels)
    # Extend color palette to support many conditions
    color_palette = ['rgb(240,240,240)', 'rgb(254,224,144)', 'rgb(253,141,60)', 'rgb(240,59,32)', 
                     'rgb(117,107,177)', 'rgb(49,130,189)', 'rgb(49,163,84)', 'rgb(99,99,99)'][::-1]
    colorscale = []
    for i in range(max_conditions + 1):
        val = i / max_conditions if max_conditions > 0 else 0
        color = color_palette[i % len(color_palette)]
        colorscale.append([val, color])

    fig = go.Figure(data=go.Heatmap(
        x=vz_vals,
        y=mu_vals,
        z=Z_2d,
        text=hover_text_2d,
        hoverinfo="text", 
        colorscale=colorscale,
        colorbar=dict(
            title="Conditions Met",
            tickvals=list(range(max_conditions + 1)),
            ticktext=[str(i) for i in range(max_conditions + 1)]
        ),
        showscale=True
    ))

    fig.update_layout(
        title="Interactive Heatmap: Overlapping Conditions in Parameter Space",
        xaxis_title="V_z (meV)",
        yaxis_title="mu (meV)",
        xaxis_range=[0.0, max(vz_vals) if len(vz_vals)>0 else 1.2],
        yaxis_range=[0, max(mu_vals) if len(mu_vals)>0 else 4.5],
        width=800,
        height=800,
        hovermode='closest'
    )

    fig.show()

In [21]:
# ---------- Helper Functions ----------
def has_peaks(one_array, return_float = False):
    pks = find_peaks(one_array)[0]
    lnpks = len(pks)
    if return_float:
        if lnpks == 0:
            return 0.0
        return float(pks[0])
    else:
        return lnpks != 0

def has_negative_peaks(one_array):
    return np.asarray(len(find_peaks(-one_array)[0]) != 0)

def has_max_symm(one_arr):
    ismax_1 = all(one_arr <= one_arr[0])
    return ismax_1 
    
def get_cpeak_dists(cond_arr1, cond_arr2):
    pks1 = np.asarray([has_peaks(cond_arr1[i,:], return_float=True) for i in range(cond_arr2.shape[0])])
    pks2 = np.asarray([has_peaks(cond_arr2[i,:], return_float=True) for i in range(cond_arr2.shape[0])])
    ret = np.asarray([[pks1[i], pks2[i]] for i in range(cond_arr2.shape[0])])
    return ret

def has_peaks_in_window(peak_data, window):
    pos_peak_eng = peak_data[2]
    neg_peak_eng = peak_data[4]
    return (np.abs(pos_peak_eng) <= window) and (np.abs(neg_peak_eng) <= window)
    
def has_symmetric_peaks(peak_data):
    pos_peak_eng = peak_data[2]
    neg_peak_eng = peak_data[4]
    return np.isclose(np.abs(pos_peak_eng) - np.abs(neg_peak_eng), 0)

# ---------- Condition Functions ----------
def check_peak_window(peaks_data_left, peaks_data_right, window):
    condition  = lambda i: has_peaks_in_window(peaks_data_left[i,:], window) \
                            and has_peaks_in_window(peaks_data_right[i,:], window)
    return np.asarray([condition(i) for i in range(peaks_data_right.shape[0])])

def check_peak_symmetry(peaks_data_left, peaks_data_right):
    condition  = lambda i: has_symmetric_peaks(peaks_data_left[i,:]) \
                            and has_symmetric_peaks(peaks_data_right[i,:])
    return np.asarray([condition(i) for i in range(peaks_data_right.shape[0])])
    
def check_arr_monotonic(cond_arr1, cond_arr2):
    is_monotonic_arr = np.asarray([(np.all(np.diff(cond_arr1[i,:])<=0) &
                        np.all(np.diff(cond_arr2[i,:])<=0)).astype(int) for i in range(cond_arr2.shape[0])])
    return is_monotonic_arr
    
def check_max_at_symm(cond_arr1, cond_arr2):
    condition = lambda i: has_max_symm(cond_arr1[i,:]) and has_max_symm(cond_arr1[i,:])
    ret = np.asarray([condition(i) for i in range(cond_arr2.shape[0])]).astype(int)
    return ret
    
def check_resonance_peak(cond_arr1, cond_arr2):
    return np.asarray([has_peaks(cond_arr1[i,:]) and has_peaks(cond_arr2[i,:]) for i in range(cond_arr2.shape[0])]).astype(int)

def check_negative_peaks(cond_arr1, cond_arr2):
    return np.asarray([has_negative_peaks(cond_arr1[i,:]) or has_negative_peaks(cond_arr2[i,:]) for i in range(cond_arr2.shape[0])]).astype(int)

def check_correlation(cond_arr1, cond_arr2, corr_thresh):
    corrs = np.asarray([hp.calc_correlation(cond_arr1[i,:], cond_arr2[i,:]) for i in range(cond_arr2.shape[0])])
    corrs = np.asarray([1.0 if corr > 1 else corr for corr in corrs])
    corrs = np.asarray([0.0 if corr < 0.0 else corr for corr in corrs])
    corrs = np.asarray([0.0 if corr < corr_thresh else corr for corr in corrs])
    corrs = np.asarray([1.0 if corr > corr_thresh else corr for corr in corrs])
    return corrs


In [23]:
# Define the data dictionary mapping argument names to arrays and threshold values
data_dict = {
    'mu': mu,
    'V_z': V_z,
    'cond_arr1': brcl,
    'cond_arr2': brcr,
    'peaks_data_left': peaks_left,
    'peaks_data_right': peaks_right,
    'window': 0.015,         # Threshold parameter for check_peak_window
    'corr_thresh': 0.9     # Threshold parameter for check_correlation
}

# Define conditions dictionary.
# Note how we use {window} and {corr_thresh} in the string keys so the hover text is dynamically formatted!
conditions_dict = {
    'Peaks in Window (±{window} eV)': check_peak_window,
    'Symmetric Peaks': check_peak_symmetry,
    'Monotonic Decreasing': check_arr_monotonic,
    'Max Conductance at Symm': check_max_at_symm,
    'Resonance Peaks': check_resonance_peak,
    'Negative Peaks': check_negative_peaks,
    'High Correlation (>{corr_thresh})': check_correlation
}

# Generate the heatmap
plot_condition_heatmap(conditions_dict, data_dict)